In [9]:
import pandas as pd
import os
from tqdm import tqdm

In [24]:
# For reference, columns in main dataset:
ref_cols = pd.read_csv('/notebooks/data/cell_table_transformed.csv', nrows=0, index_col=0).columns.tolist()
ref_cols

['SMA',
 'CA9',
 'nuclei',
 'CD98',
 'CytC',
 'CD4',
 'MSH2',
 'FoxP3',
 'MCT1',
 'PDL1',
 'CD31',
 'PD1',
 'ASCT2',
 'CD7',
 'LDH',
 'CD163',
 'STING1',
 'DCN',
 'CD68',
 'GS',
 'CD8',
 'CD3e',
 'GLS',
 'MPO',
 'ATP5A',
 'CS',
 'CD20',
 'PKM2',
 'GLUT1',
 'HLADRa',
 'CD14',
 'CD45',
 'PanCK',
 'MSH6',
 'ARG1',
 'CPT1A',
 'Vimentin',
 'Ki67',
 'membrane',
 'centroid-0',
 'centroid-1',
 'eccentricity',
 'perimeter',
 'convex_area',
 'area',
 'axis_major_length',
 'axis_minor_length',
 'label',
 'fov']

In [37]:
X[X['area'] * X['nuclei'] >= 0]

,CD4,CD20,FoxP3,CD14,GLUT1,Au,nuclei,COL1A1,CD68,CS,...,centroid-0,centroid-1,eccentricity,perimeter,convex_area,area,axis_major_length,axis_minor_length,label,fov
1,0.001435,0.001390,0.002338,0.001435,0.002495,200.266529,0.027020,0.002016,0.001711,0.002963,...,3.710744,16.227273,0.954321,71.556349,254.0,242.0,32.729185,9.778908,1,CRC-IFN_ACCC_15714_TC_3
2,0.015238,0.000480,0.002695,0.002410,0.002976,40.576479,0.005874,0.015290,0.001490,0.001492,...,3.691919,80.348485,0.925470,63.384776,214.0,198.0,26.993677,10.225779,2,CRC-IFN_ACCC_15714_TC_3
3,0.000983,0.001106,0.000814,0.000548,0.000870,70.866114,0.009893,0.004825,0.000501,0.001038,...,11.850060,175.220073,0.801656,113.840620,867.0,827.0,42.323655,25.300501,3,CRC-IFN_ACCC_15714_TC_3
6,0.001717,0.000699,0.000631,0.001378,0.001333,5.905698,0.005446,0.002105,0.007731,0.001537,...,8.308057,301.798578,0.739265,81.597980,447.0,422.0,28.688676,19.319365,6,CRC-IFN_ACCC_15714_TC_3
7,0.001004,0.000399,0.001210,0.001276,0.001481,1.843052,0.018313,0.004997,0.000648,0.000956,...,9.670886,327.987342,0.741835,93.840620,590.0,553.0,32.742285,21.956416,7,CRC-IFN_ACCC_15714_TC_3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5588,0.000149,0.000194,0.000458,0.001089,0.000275,114.874168,0.003490,0.002722,0.001142,0.000415,...,2042.065868,722.383234,0.730726,48.142136,176.0,167.0,17.762381,12.125865,5588,CRC-IFN_ACCC_15714_TC_3
5589,0.000187,0.000121,0.000269,0.000173,0.000000,1.764400,0.002917,0.000093,0.000177,0.000181,...,2042.549669,1578.927152,0.768593,49.692388,163.0,151.0,17.929684,11.470293,5589,CRC-IFN_ACCC_15714_TC_3
5590,0.000121,0.000107,0.000304,0.000058,0.000082,2.092443,0.003729,0.000382,0.000138,0.000707,...,2042.456311,1610.849515,0.882027,59.313708,223.0,206.0,24.196334,11.401282,5590,CRC-IFN_ACCC_15714_TC_3
5591,0.000216,0.000122,0.000225,0.000047,0.000164,1.986577,0.003564,0.000122,0.000096,0.000676,...,2042.326923,1593.403846,0.715322,46.485281,162.0,156.0,17.073186,11.930659,5591,CRC-IFN_ACCC_15714_TC_3


In [46]:
seg_folder = '/notebooks/data/sampling/data/'
aggX = None # Aggregated cell table

# Some columns will be renamed
cols_to_rename = {
    'HH3_dsDNA': 'nuclei',
    'ASCT2CAL33_ASCT2D7C12': 'ASCT2',
    'HLA1': 'membrane',
    'AKT (pan)': 'AKT',
}

for fov in tqdm(os.listdir(seg_folder)):
    table_path = seg_folder + fov + '/cell_table.csv'
    assert os.path.exists(table_path) # Cell table exists

    X = pd.read_csv(table_path, index_col=0)

    # Filter cells as in FilterAndViz_Hamburg.ipynb
    X = X[X['area'] > 120]
    X = X[X['area'] < 1500]
    X = X[X['area'] * X['HH3_dsDNA_clahe.tiff'] > 2e3]
    X = X[X['Au.tiff'] < 1000]

    cols_to_exclude = [c for c in X.columns if "_clahe.tiff" in c]
    X.drop(columns=cols_to_exclude, inplace=True)

    # Drop '.tiff' from column names
    X.columns = [c.replace('.tiff', '') for c in X.columns]

    # Rename columns
    X.rename(columns=cols_to_rename, inplace=True)

    # Keep track of fov of origin
    X['fov'] = fov    

    # If this is not the first fov aggregated:
    if aggX is not None:
        aggX = pd.concat([aggX, X], axis=0, ignore_index=True)
    else: # First fov aggregated, initialize and print details
        print(X.columns)
        print([c for c in X.columns if c not in ref_cols])
        print([c for c in ref_cols if c not in X.columns])
        aggX = X.copy()

aggX.to_csv("/notebooks/data/sampling/cell_table.csv")

  3%|██▋                                                                               | 5/156 [00:00<00:03, 43.16it/s]

Index(['CD4', 'CD20', 'FoxP3', 'CD14', 'GLUT1', 'Au', 'nuclei', 'COL1A1',
       'CD68', 'CS', 'CytC', 'LDH', 'MSH2', 'BRCA1', 'CD31', 'GLS', 'Ki67',
       'ASCT2', 'MCT1', 'pSMAD3', 'ATP5A', 'PKM2', 'CD3e', 'RUNX2',
       'Calprotectin', 'STING1', 'CPT1A', 'Tox', 'CD98', 'MX1', 'PanCK_EpCAM',
       'SMA', 'HNF4a', 'PDL1', 'CD8', 'CD45', 'membrane', 'AKT', 'CD36',
       'CD163', 'Vimentin', 'centroid-0', 'centroid-1', 'eccentricity',
       'perimeter', 'convex_area', 'area', 'axis_major_length',
       'axis_minor_length', 'label', 'fov'],
      dtype='object')
['Au', 'COL1A1', 'BRCA1', 'pSMAD3', 'RUNX2', 'Calprotectin', 'Tox', 'MX1', 'PanCK_EpCAM', 'HNF4a', 'AKT', 'CD36']
['CA9', 'PD1', 'CD7', 'DCN', 'GS', 'MPO', 'HLADRa', 'PanCK', 'MSH6', 'ARG1']


100%|████████████████████████████████████████████████████████████████████████████████| 156/156 [00:05<00:00, 26.35it/s]


In [47]:
# Remaining cells per fov after filtering
aggX.groupby('fov').size().sort_values(ascending=False)

fov
CRC-IFN_ACCC_17481_IM_3    6682
CRC-IFN_ACCC_13897_IM_2    6508
CRC-IFN_ACCC_7451_IM_1     6336
CRC-IFN_ACCC_1146_TC_2     6195
CRC-IFN_ACCC_7451_IM_3     6102
                           ... 
CRC-IFN_ACCC_7451_TC_3     2564
CRC-IFN_ACCC_6802_IM_1     2494
CRC-IFN_ACCC_4267_IM_3     2492
CRC-IFN_ACCC_16920_IM_1    2272
CRC-IFN_ACCC_6802_IM_2     2155
Length: 156, dtype: int64